# 02 Messung — Photometrie, Lichtkurve, Periode

**Ergebnis:** `messtabelle.csv` und `zeitreihe.csv` im Ergebnisordner der Sitzung, dazu Lichtkurve und Periodensuche.

| Schritt 1 — Formular starten | Schritt 2 — Prüfen | Schritt 3 — Automatisch |
|:---|:---|:---|
| Zelle **direkt darunter** ausführen (▶ oder Shift+Enter) → Widgets erscheinen | Meist Defaults übernehmen; Hinweisfarbe oben: **grün** = ok · **orange** = prüfen · **rot** = Fehler | Zelle **„Ab hier automatisch"** → ▶ → **„Diese Zelle und alle folgenden"** |

**Ihr Fall (aus NB00/01):** `A` Mono + Standardfilter → ein Lauf, dann NB03 · `B/C/D` Farbe/RGB → je Kanal ein Lauf (`VARPHOT_TAG_B/G/R`), optional Transformation (Zelle unten), dann NB03

### Felder im Überblick

| Feld | Testlauf (LCO) | Eigene Aufnahmen |
|:---|:---|:---|
| **WCS je Bild** | *automatisch* lassen | *automatisch* (Siril-Bilder → Registrierung) |
| **Apertur** | *aus dem Plan (NB00)* lassen | *aus dem Plan (NB00)* lassen |
| **Kernmaske [px]** | 1,5 lassen | 1,5 bei gesättigtem Ziel; sonst 0 |
| **Periode von/bis** | leer lassen | leer = automatisch; bei bekannter Periode eingrenzen |
| **Teleskopöffnung [m]** | 0,4 eingetragen | Ihre Öffnung (Szintillationsterm); leer = ohne |

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from tqdm.auto import tqdm

from varphot import Session, load_plan
from varphot import workflow as W
from varphot.hints import bind, nb02_hints

SESS = Session.current()
PLAN = load_plan(SESS.plan) if SESS.plan and Path(SESS.plan).exists() else None
assert SESS.field_config and Path(SESS.field_config).exists(), "Keine field_config.json in der Sitzung — zuerst 01_vorbereitung.ipynb ausführen."
print(f"Sitzung {SESS.workspace}\n  Ziel {SESS.target} · Methode {SESS.method} · Referenz {Path(SESS.ref_frame).name} · Ergebnisse {SESS.results_dir()}")

_st, _lay = {"description_width": "170px"}, widgets.Layout(width="640px")
w_wcs = widgets.Dropdown(description="WCS je Bild:", options=[("automatisch: Kopf, sonst Registrierung", "auto"), ("nur aus dem Bildkopf", "header"), ("immer Registrierung auf die Referenz", "register")],
                         value=os.environ.get("VARPHOT_WCS_MODE") or SESS.wcs_mode or "auto", style=_st, layout=_lay)
w_ap = widgets.Dropdown(description="Apertur:", options=[("aus dem Plan (NB00)" if PLAN and PLAN.aperture else "aus dem Referenzbild (Wachstumskurve/S/N)", "auto"), ("manuell (Radius unten)", "manual")], value=os.environ.get("VARPHOT_AP_MODE", "auto"), style=_st, layout=_lay)
w_apr = widgets.IntText(description="Radius [px] (manuell):", value=int(os.environ.get("VARPHOT_AP_R") or (PLAN.aperture.get("ap_r", 8) if PLAN and PLAN.aperture else 8)), style=_st, layout=_lay)
w_core = widgets.FloatText(description="Kernmaske [px], 0 = aus:", value=float(os.environ.get("VARPHOT_CORE_R", 1.5)), style=_st, layout=_lay)
w_pmin = widgets.Text(description="Periode von [d]:", value=os.environ.get("VARPHOT_PMIN", ""), placeholder="leer = automatisch", style=_st, layout=_lay)
w_pmax = widgets.Text(description="Periode bis [d]:", value=os.environ.get("VARPHOT_PMAX", ""), placeholder="leer = halbe Zeitbasis", style=_st, layout=_lay)
w_aperture_m = widgets.Text(description="Teleskopöffnung [m]:", value=os.environ.get("VARPHOT_APERTURE_M", "0.4" if SESS.source == "lco-demo" else ""), placeholder="leer = ohne Szintillationsterm", style=_st, layout=_lay)
w_workers = widgets.IntSlider(description="Prozesse:", value=4, min=1, max=8, style=_st, layout=_lay)
w_hint = widgets.HTML()
bind(w_hint, [w_wcs, w_ap, w_apr, w_core, w_pmin, w_pmax, w_aperture_m],
     lambda: nb02_hints(w_wcs.value, w_ap.value, w_apr.value, w_core.value, w_pmin.value, w_pmax.value, w_aperture_m.value, bool(PLAN and PLAN.aperture)))
display(widgets.VBox([w_hint, w_wcs, w_ap, w_apr, w_core, w_pmin, w_pmax, w_aperture_m, w_workers]))

## Ab hier automatisch

*Ausführen → Diese Zelle und alle folgenden.* Beim ersten Lauf dauert die Photometrie je nach Bildzahl und Registrierung Sekunden bis Minuten; ein zweiter Lauf mit unveränderten Bildern nutzt den Zwischenspeicher.

In [ ]:
AP = None if w_ap.value == "auto" else {"ap_r": w_apr.value, "ap_r_in": w_apr.value + 3.0, "ap_r_out": int(np.ceil(np.sqrt((w_apr.value + 3.0) ** 2 + 1.5 * w_apr.value ** 2)))}
R = W.measure(SESS, PLAN, wcs_mode=w_wcs.value, ap=AP, core_r=w_core.value or None, workers=w_workers.value,
              progress=lambda it, total: tqdm(it, total=total), aperture_m=float(w_aperture_m.value) if w_aperture_m.value.strip() else None)
LC, FR = R["lc"], R["frames"]
for n in R["notes"]:
    print("•", n)
print(f"Apertur {R['ap']} px · WCS-Modus {R['wcs_mode']}" + (" · Kernmaske für das gesättigte Ziel angewandt" if R["core_used"] else ""))
print("Bildqualität:", R["flags"])
if "wcs_source" in FR:
    print("WCS-Quelle:", FR["wcs_source"].value_counts().to_dict())
if len(LC) < 8:
    raise RuntimeError(f"Nur {len(LC)} brauchbare Bilder — Bildqualität/WCS prüfen (Tabelle unten), ggf. WCS-Modus oder Referenz ändern.")
display(FR[FR["frame_flags"] != ""].head(12))

In [ ]:
try:
    PER = W.analyse_period(LC, float(w_pmin.value) if w_pmin.value.strip() else None, float(w_pmax.value) if w_pmax.value.strip() else None)
    print(f"Lomb-Scargle-Periode {PER['period']:.4f} d (weitere Maxima: " + ", ".join(f"{p:.3f} d" for p, _ in PER["top"][1:]) + f"); Fourier-Anpassung: Periode {PER['harmonic']['period']:.4f} d, Amplitude {PER['harmonic']['amplitude']:.2f} mag, Residuum {PER['harmonic']['rms']:.3f} mag")
except ValueError as e:
    PER = None
    print("Periodensuche übersprungen:", e)
t0 = LC["bjd_tdb"].where(LC["bjd_tdb"].notna(), LC["jd_mid"]).to_numpy()
y = LC["mag"].to_numpy()
fig, ax = plt.subplots(1, 3 if PER else 2, figsize=(16 if PER else 11, 3.8))
ax[0].errorbar(t0 - t0.min(), y, LC["mag_err"], fmt=".", ms=4, alpha=0.8)
if "target_sat" in LC:
    s = LC["target_sat"].to_numpy(bool); ax[0].plot((t0 - t0.min())[s], y[s], "o", mfc="none", mec="C3", ms=6, label="Ziel gesättigt"); ax[0].legend()
ax[0].invert_yaxis(); ax[0].set_xlabel("Tage seit Beginn"); ax[0].set_ylabel("mag"); ax[0].set_title(f"{SESS.target} — Lichtkurve ({len(LC)} Punkte)")
if "check_minus_cat" in LC:
    ax[1].plot(t0 - t0.min(), LC["check_minus_cat"], ".", color="darkorange"); ax[1].axhline(0, color="k", lw=0.5); ax[1].set_title(f"Check − Katalog (σ = {LC['check_minus_cat'].std():.3f} mag)"); ax[1].set_xlabel("Tage seit Beginn")
if PER:
    P = PER["period"]; ph = ((t0 - t0.min()) / P) % 1
    ax[-1].plot(np.r_[ph, ph + 1], np.r_[y, y], ".", ms=4); ax[-1].invert_yaxis(); ax[-1].set_title(f"Phase (P = {P:.3f} d)"); ax[-1].set_xlabel("Phase")
plt.tight_layout(); plt.show()

In [ ]:
good = int((FR["frame_flags"] == "").sum())
rows = [("Bilder brauchbar", good >= 0.5 * len(FR), f"{good} von {len(FR)}  {R['flags']}"),
        ("Lichtkurve", len(LC) >= 20, f"{len(LC)} Punkte, Amplitude {LC['mag'].max() - LC['mag'].min():.2f} mag, mittlerer Fehler {LC['mag_err'].median():.3f} mag"),
        ("Check-Stern", ("check_minus_cat" in LC) and LC["check_minus_cat"].std() < 0.05, (f"σ {LC['check_minus_cat'].std():.3f} mag, mittlere Abweichung {LC['check_minus_cat'].mean():+.3f} mag" if "check_minus_cat" in LC else "kein Check-Stern")),
        ("Periode", PER is not None, f"{PER['period']:.4f} d" if PER else "nicht bestimmt")]
html = "".join(f'<tr><td style="padding:3px 10px;color:{"#4caf50" if ok else "#ff9800"};font-weight:bold">{"✓" if ok else "⚠"}</td><td style="padding:3px 10px">{a}</td><td style="padding:3px 10px;color:#888">{b}</td></tr>' for a, ok, b in rows)
from IPython.display import HTML
display(HTML(f'<table style="border-collapse:collapse;font-family:monospace;font-size:13px"><tr style="background:#1e1e2e;color:#cdd6f4"><th colspan="3" style="padding:6px 14px;text-align:left">Ergebnis Messung</th></tr>{html}</table>'))
print("Gespeichert:", SESS.results_dir() / "zeitreihe.csv", "\n→ weiter mit 03_bericht.ipynb (AAVSO-Bericht); bei Farbkamera/RGB-Filter vorher die Transformation unten.")

## Optional: Transformation anwenden (Farbkamera, RGB-Filter)

Nur mit Koeffizienten aus **NB08** und den Messtabellen der **drei Kanäle** (je ein Lauf dieses Notebooks pro Kanal; die Ergebnisordner der Kanäle stehen in `VARPHOT_TAG_B/G/R`). Je Bild und Comp wird das Ziel transformiert (Handbuch Gl. 6.11–6.14) und über die Comps gemittelt;
dazu der Check-Stern. Comps brauchen **AAVSO-Werte** für B und V (Gaia-abgeleitete werden abgelehnt). Ergebnis `zeitreihe_transformiert.csv` für NB03. Ohne `VARPHOT_TRANSFORM_FILE` passiert nichts.

> **TODO — Erstmessung der Transformationskoeffizienten steht noch aus:**
> NB08 läuft bisher nur im Demo-Modus (synthetische Koeffizienten). Für echte transformierte Meldungen (`TRANS=YES`) müssen zuerst
> eigene Aufnahmen eines AAVSO-Standardfelds (M 67 empfohlen, günstig April–Mai; alternativ NGC 7790 im Herbst) mit der
> Ares-C Pro bei Gain 0, LRN ausgewertet werden:
> 1. Photometrische Nacht → Standardfeld in B, G, R bei ≥ 2 Luftmassen aufnehmen (≥ 10 Comps, (B−V) = 0,3–1,5)
> 2. Siril CFA → Kanäle trennen → NB04 je Kanal → Messtabellen
> 3. NB08 `MODE="real"` ausführen → `transform_ares-c-pro_gain-0-lrn.json` in `PHOTOMETRIE_PY_DATA/`
> 4. `VARPHOT_TRANSFORM_FILE` auf diese Datei setzen und dieses Notebook erneut ausführen
>
> Details: KONZEPT.md §11.8 „Workflow: Erstmessung der Transformationskoeffizienten mit M 67"

In [ ]:
TRANSFORM_FILE = os.environ.get("VARPHOT_TRANSFORM_FILE")
if not TRANSFORM_FILE:
    print("Keine Transformation (VARPHOT_TRANSFORM_FILE nicht gesetzt) — Zeitreihe bleibt untransformiert (V bzw. TG/TB/TR).")
else:
    from varphot import TransformSolution, transformed_with_check, maturity_warnings, read_measurements, stars_from_field_config
    SOL = TransformSolution.load(TRANSFORM_FILE)
    base = SESS.results_dir().parent
    CH = {"b": base / os.environ["VARPHOT_TAG_B"] / "messtabelle.csv", "g": base / os.environ["VARPHOT_TAG_G"] / "messtabelle.csv", "r": base / os.environ["VARPHOT_TAG_R"] / "messtabelle.csv"}
    MEAS = {ch: read_measurements(f) for ch, f in CH.items() if f.exists()}
    record = json.loads(Path(SESS.field_config).read_text())
    STARS_T = stars_from_field_config(record)
    STARS_T["B"] = [next((s["B"] for s in record["sterne"] if str(s["label"]) == i), np.nan) for i in STARS_T["id"]]
    STARS_T["R"] = [next((s.get("R") for s in record["sterne"] if str(s["label"]) == i), np.nan) for i in STARS_T["id"]]
    STARS_T["source"] = ["Gaia" if str(next((s["auid"] for s in record["sterne"] if str(s["label"]) == i), "")).startswith("Gaia") else "AAVSO" for i in STARS_T["id"]]
    CHECK_ID = next((i for i, r in zip(STARS_T["id"], STARS_T["role"]) if r == "check"), None)
    LC_TR = transformed_with_check(MEAS, STARS_T, "target", SOL, check_id=CHECK_ID)
    if "V_check" in LC_TR:
        _ck = STARS_T.set_index("id").loc[CHECK_ID]; _d = LC_TR["V_check"].mean() - float(_ck["V"])
        print(f"Check-Stern {CHECK_ID}: transformiert V = {LC_TR['V_check'].mean():.3f} (Katalog {float(_ck['V']):.3f}, Differenz {_d:+.3f} mag, Streuung {LC_TR['V_check'].std():.3f})")
        if abs(_d) > 0.05:
            print("⚠ Check-Stern weicht > 0,05 mag vom Katalog ab — Transformation/Comps prüfen, nicht melden")
    LC_TR.to_csv(SESS.results_dir() / "zeitreihe_transformiert.csv", index=False)
    for _w in maturity_warnings(["transform_fit", "transform_lightcurve"]):
        print("⚠", _w)
    print(f"transformiert: {len(LC_TR)} Bilder; Gültigkeit B−V {SOL.bv_range[0]:.2f}…{SOL.bv_range[1]:.2f}; Ziel bei B−V ≈ {LC_TR['BV'].median():.2f}")